# Using ``run_ep()`` to directly run simulations with EnergyPlus

``run_ep()`` (``runEp()`` before accim 1.0.0) is based on a script by eppy's development team (https://eppy.readthedocs.io/en/latest/runningeplus.html, section 'Running in parallel processes using Generators'), with some changes. You should check out eppy as well: it is an excellent package.

``run_ep()`` takes the EPW and IDF files in the folder where it is run, simulates each IDF with each EPW, and names the output files following the pattern "IDFname[EPWname".

The IDF in this folder is an output IDF of ``AddAccis()`` for EnergyPlus 25.1, so EnergyPlus 25.1 must be installed (in ``C:\EnergyPlusV25-1-0``) to run this notebook.

Since accim 1.0.0, ``run_ep()`` can also run the simulations through the Python API of EnergyPlus (EnergyPlus 24.2 or later) with ``backend='api'``, with the same results: see the section *Running the simulations with the Python API of EnergyPlus* of the Detailed use page.

First of all, let's see what files we do have in the folder:

In [ ]:
import os
original_files = [i for i in os.listdir()]
print(*original_files, sep='\n')

As you can see, the EPW file or files we have is or are:

In [ ]:
print([i for i in original_files if i.endswith('.epw')])

And the IDF file or files we have is or are:

In [ ]:
print([i for i in original_files if i.endswith('.idf')])

Before running the simulations, let's see how much disk space their results will take. Since accim 1.1.0, ``estimate_results_size()`` (in ``accim.run``) takes the IDFs and EPWs of the folder, as ``run_ep()`` takes them, and estimates the size of the results of their simulations from the outputs of the IDFs (the values that their ``Output:Variable`` and ``Output:Meter`` objects report, and how often), without simulating them, in a few seconds. This is useful before running hundreds or thousands of simulations, which could fill the disk:

In [ ]:
from accim.run import estimate_results_size
estimated_size = estimate_results_size(only_accim_idfs=True)

It also returns a table with a row for each IDF, with the sizes in MB: those of the files of each simulation (``eso_mb``, ``csv_mb``, ``mtr_mb``, ``meter_csv_mb``, ``sql_mb`` and ``other_mb``) and of the whole simulation (``simulation_mb``), that of the files that ``remove_files()`` keeps (``kept_mb``), the memory needed to read the values of the CSV file (``csv_memory_mb``), and the size of all the simulations of the IDF (``total_mb`` and ``total_kept_mb``):

In [ ]:
estimated_size

With ``simulate=True``, it measures the size instead: it simulates one IDF of each group of IDFs with the same outputs (for instance, all the output IDFs of a model of ``AddAccis()``) with the first EPW, as ``run_ep()`` runs it, in a temporary folder. It takes as long as these simulations, but gives the size of their results (other EPWs give about the same size):

In [ ]:
measured_size = estimate_results_size(simulate=True, only_accim_idfs=True)

Based on that, let's use ``run_ep()``:

In [ ]:
from accim.run import run
run.run_ep(
    only_accim_idfs=True, #only runs output IDFs, that is, IDFs with "[" in its name
    confirm=True, #to skip confirmation
    processes=2, #to specify the number of CPUs to be used
    energyplus_version='25.1', #the EnergyPlus version of the IDFs, which is also the version of EnergyPlus used to run them
)

``run_ep()`` just informed us about the input files, and the number of simulation runs and name of each one. So now, let's see what are the outputs:

In [ ]:
new_files = [i for i in os.listdir() if not(any(i in j for j in original_files))]
print(*new_files, sep='\n')

Let's compare the size of these files with the estimate and the measurement of ``estimate_results_size()``:

In [ ]:
size = sum(os.path.getsize(i) for i in new_files) / 1e6
print(f'The files of the simulation: {size:.1f} MB')
print(f"Estimated from the outputs of the IDF: {estimated_size['total_mb'].sum():.1f} MB")
print(f"Measured in a simulation: {measured_size['total_mb'].sum():.1f} MB")

The measurement in a simulation is the size of the files of the simulation, and the estimate from the outputs of the IDF, without simulating it, is close to it (within about 10 % for results larger than 5 MB): enough to choose the outputs, the number of simulations or the disk before running them.

The CSV files we have generated follow the naming convention 'IDF[EPW' and therefore are suitable to be used with the class ``Table``, in the module ``accim.data.postprocessing.main`` (see the notebook about ``Table``). These are the CSV files with the results (the files ending in 'Table.csv' and 'Zsz.csv' are other reports of EnergyPlus):

In [ ]:
print(*[i for i in os.listdir() if i.endswith('.csv') and not(i.endswith('Table.csv')) and not (i.endswith('Zsz.csv'))], sep='\n')

Now, let's remove all output files of the simulation run:

In [ ]:
for i in new_files:
    os.remove(i)